# Step 3 — Pretrained Models (Transfer Learning)
**Oral Diseases Image Classification · Sprint Two**

We train **three ImageNet-pretrained models** on exactly the same data, split, augmentation and class weights as the custom CNN, so the comparison is fair:
`ResNet50` · `EfficientNetB0` · `MobileNetV2`

**Training recipe (same for all three)**
1. **Phase 1 – head only:** the pretrained backbone is frozen, only the new classifier head learns (LR 1e-3).
2. **Phase 2 – fine-tuning:** the last 30 backbone layers are unfrozen and trained with a small LR (1e-5). BatchNorm layers stay frozen (standard practice for small datasets).
3. Each model gets **its own correct preprocessing** (ResNet50 = caffe style, MobileNetV2 = [-1, 1], EfficientNetB0 = raw 0-255 because it normalises internally). Using the wrong one silently hurts accuracy.
4. Early stopping and the best-epoch choice use the **validation** set. The **test set is not touched** here.

**Before running**
1. *Settings → Accelerator → **GPU T4 x2***
2. *Settings → **Internet → ON*** (needed once to download the ImageNet weights; Kaggle may ask you to verify your phone number)
3. *Add Input → Notebooks → Your Work →* the **Step 1** notebook (same as Step 2)

⏱ Rough estimate for all three models: 40–70 minutes. Each model's result is saved as soon as it finishes.

In [ ]:
import os, gc, json, time, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50, EfficientNetB0, MobileNetV2
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, accuracy_score

SEED, IMG_SIZE, BATCH = 42, (224, 224), 32
EPOCHS_HEAD   = int(os.environ.get("EPOCHS_HEAD", 8))
EPOCHS_FT     = int(os.environ.get("EPOCHS_FT", 25))
LR_HEAD, LR_FT = 1e-3, 1e-5
UNFREEZE_LAST = 30
DROPOUT       = 0.3
WEIGHTS       = None if os.environ.get("WEIGHTS") == "none" else "imagenet"
MODELS_TO_RUN = ["ResNet50", "EfficientNetB0", "MobileNetV2"]     # edit this list to run fewer models

SEARCH_ROOT = os.environ.get("DATA_ROOT", "/kaggle/input")
WORK_DIR    = os.environ.get("WORK_DIR", "/kaggle/working")

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

DATA_DIR = None
for d, _, files in os.walk(SEARCH_ROOT):
    if "manifest.csv" in files and "meta.json" in files:
        DATA_DIR = d; break
assert DATA_DIR, "Step-1 output not found. Add it via: Add Input -> Notebooks -> Your Work"
CLASS_NAMES = json.load(open(os.path.join(DATA_DIR, "meta.json")))["class_names"]
for sub in ("results", "models"): os.makedirs(os.path.join(WORK_DIR, sub), exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPUs      :", tf.config.list_physical_devices("GPU") or "NONE  <-- turn on the GPU accelerator!")
print("DATA_DIR  :", DATA_DIR)
print("Classes   :", CLASS_NAMES)

## 1) Data (identical to Step 2)

In [ ]:
def make_ds(split, shuffle):
    ds = keras.utils.image_dataset_from_directory(
        os.path.join(DATA_DIR, split), label_mode="categorical", class_names=CLASS_NAMES,
        image_size=IMG_SIZE, batch_size=BATCH, shuffle=shuffle, seed=SEED)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = make_ds("train", True)
val_ds   = make_ds("val",   False)

counts  = np.array([len(os.listdir(os.path.join(DATA_DIR, "train", c))) for c in CLASS_NAMES])
CLASS_WEIGHT = {i: float(w) for i, w in enumerate(counts.sum() / (len(counts) * counts))}
print(pd.DataFrame({"class": CLASS_NAMES, "train_images": counts,
                    "class_weight": [round(CLASS_WEIGHT[i], 2) for i in range(len(CLASS_NAMES))]}).to_string(index=False))

## 2) Model builders

In [ ]:
@keras.utils.register_keras_serializable(package="oral")
class ImageNetPreprocess(layers.Layer):
    """Model-specific input preprocessing, kept INSIDE the model so the deployed app needs no extra code."""
    def __init__(self, mode="none", **kwargs):
        super().__init__(**kwargs); self.mode = mode
    def call(self, x):
        if self.mode == "none":
            return x
        return keras.applications.imagenet_utils.preprocess_input(x, mode=self.mode)
    def get_config(self):
        cfg = super().get_config(); cfg.update(mode=self.mode); return cfg

ZOO = {                       # name: (constructor, preprocessing mode)
    "ResNet50":       (ResNet50,       "caffe"),
    "EfficientNetB0": (EfficientNetB0, "none"),
    "MobileNetV2":    (MobileNetV2,    "tf"),
}

def build_model(name):
    ctor, mode = ZOO[name]
    try:
        base = ctor(include_top=False, weights=WEIGHTS, input_shape=IMG_SIZE + (3,))
    except Exception as e:
        raise RuntimeError("Could not load ImageNet weights. Turn Settings -> Internet ON and re-run.") from e
    base.trainable = False
    inp = keras.Input(IMG_SIZE + (3,))
    x = layers.RandomFlip("horizontal")(inp)             # same augmentation as the custom CNN
    x = layers.RandomRotation(0.08)(x)
    x = layers.RandomZoom(0.1)(x)
    x = layers.RandomContrast(0.1)(x)
    x = ImageNetPreprocess(mode)(x)
    x = base(x, training=False)                          # keep BatchNorm in inference mode
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(DROPOUT)(x)
    out = layers.Dense(len(CLASS_NAMES), activation="softmax")(x)
    return keras.Model(inp, out, name=name + "_clf"), base

def unfreeze_last(base, n):
    base.trainable = True
    for l in base.layers[:-n]:
        l.trainable = False
    for l in base.layers:                                # BatchNorm stays frozen
        if isinstance(l, layers.BatchNormalization):
            l.trainable = False

def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="categorical_crossentropy", metrics=["accuracy"])

## 3) Train + evaluate one model (function)

In [ ]:
def run_model(name):
    keras.backend.clear_session(); gc.collect(); tf.random.set_seed(SEED)
    model, base = build_model(name)
    print(f"\n{'='*70}\n{name}: {model.count_params():,} params\n{'='*70}")
    t0 = time.time()

    # ---- phase 1: head only ----
    compile_model(model, LR_HEAD)
    print(f"Phase 1 (head only) - trainable params: {sum(int(np.prod(w.shape)) for w in model.trainable_weights):,}")
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, class_weight=CLASS_WEIGHT, verbose=2, callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)])
    head_val_acc = float(model.evaluate(val_ds, verbose=0)[1])

    # ---- phase 2: fine-tune last N layers ----
    unfreeze_last(base, UNFREEZE_LAST)
    compile_model(model, LR_FT)
    print(f"\nPhase 2 (fine-tune last {UNFREEZE_LAST} layers) - trainable params: {sum(int(np.prod(w.shape)) for w in model.trainable_weights):,}")
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT, class_weight=CLASS_WEIGHT, verbose=2, callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7)])
    train_min = (time.time() - t0) / 60

    # ---- evaluate on VALIDATION ----
    y_true = np.concatenate([y.numpy().argmax(1) for _, y in val_ds])
    y_pred = model.predict(val_ds, verbose=0).argmax(1)
    t1 = time.time(); model.predict(val_ds, verbose=0); ms_img = (time.time() - t1) / len(y_true) * 1000
    acc, mf1 = accuracy_score(y_true, y_pred), f1_score(y_true, y_pred, average="macro")
    labels = list(range(len(CLASS_NAMES)))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    recall = {c: float(r) for c, r in zip(CLASS_NAMES, cm.diagonal() / np.maximum(cm.sum(1), 1))}

    path = os.path.join(WORK_DIR, "models", f"{name}_best.keras"); model.save(path)
    hist = pd.concat([pd.DataFrame(h1.history), pd.DataFrame(h2.history)], ignore_index=True)
    hist.to_csv(os.path.join(WORK_DIR, "results", f"{name}_history.csv"), index=False)

    res = {"model": name, "split": "val", "val_accuracy": float(acc), "val_macro_f1": float(mf1),
           "per_class_recall": recall, "val_acc_after_head_only": head_val_acc,
           "params": int(model.count_params()), "epochs_head": len(h1.history["loss"]), "epochs_finetune": len(h2.history["loss"]),
           "train_minutes": round(train_min, 2), "model_size_mb": round(os.path.getsize(path) / 1e6, 1),
           "ms_per_image": round(ms_img, 2), "best_val_loss": float(hist.val_loss.min())}
    json.dump(res, open(os.path.join(WORK_DIR, "results", f"{name}_val.json"), "w"), indent=2)

    # ---- report ----
    print(f"\n{name}  VALIDATION  accuracy = {acc:.3f}   macro-F1 = {mf1:.3f}   (after head only: {head_val_acc:.3f})\n")
    print(classification_report(y_true, y_pred, labels=labels, target_names=CLASS_NAMES, digits=3, zero_division=0))
    fig, ax = plt.subplots(1, 3, figsize=(18, 4.2))
    hist[["loss", "val_loss"]].plot(ax=ax[0], title=f"{name} - loss"); ax[0].axvline(len(h1.history["loss"]) - 0.5, color="gray", ls="--")
    hist[["accuracy", "val_accuracy"]].plot(ax=ax[1], title=f"{name} - accuracy (dashed = fine-tuning starts)"); ax[1].axvline(len(h1.history["loss"]) - 0.5, color="gray", ls="--")
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax[2], xticks_rotation=45, colorbar=False); ax[2].set_title(f"{name} - confusion matrix (val)")
    plt.tight_layout(); plt.show()
    return res

## 4) Run all three models

In [ ]:
all_results = {}
for name in MODELS_TO_RUN:
    all_results[name] = run_model(name)

## 5) Summary (validation set)

In [ ]:
summary = pd.DataFrame(all_results).T
cols = ["val_accuracy", "val_macro_f1", "val_acc_after_head_only", "params", "model_size_mb", "ms_per_image", "train_minutes", "epochs_finetune"]
print(summary[cols].astype({"params": int}).sort_values("val_macro_f1", ascending=False).round(3).to_string())
print("\nPer-class recall:")
print(pd.DataFrame({n: r["per_class_recall"] for n, r in all_results.items()}).round(3).to_string())